<a href="https://colab.research.google.com/github/R-SamiUllah/Flyrank-Task1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/R-SamiUllah/Flyrank-Task1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print(HF_TOKEN[:10])

hf_vbxMsbC


In [6]:
from huggingface_hub import login

login(token=HF_TOKEN)

print("Logged in successfully")

Logged in successfully


In [7]:
from datasets import load_dataset

content_ds = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content"
)

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

In [8]:
content_ds

DatasetDict({
    train: Dataset({
        features: ['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted'],
        num_rows: 519606
    })
})

In [20]:
print(content_df["search_volume"].describe())

count    376984.000000
mean        209.574544
std        3207.498742
min           0.000000
25%           0.000000
50%          10.000000
75%          20.000000
max      368000.000000
Name: search_volume, dtype: float64


In [9]:
content_df = content_ds["train"].to_pandas()

print(content_df.shape)
content_df.head()

(519606, 26)


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682.0,2555.0,None,None,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438.0,2430.0,None,None,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576.0,2645.0,None,None,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457.0,2522.0,None,None,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776.0,2552.0,None,None,True,False


In [10]:
content_df.columns.tolist()

['client_hash_id',
 'content_hash_id',
 'keyword_hash_id',
 'url_hash_id',
 'keyword_char_count',
 'keyword_token_count',
 'url_char_count',
 'content_created_date',
 'content_updated_date',
 'content_type',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'main_intent',
 'backlinks',
 'category_count',
 'keyword_created_date',
 'provider_used',
 'model_used',
 'char_count',
 'word_count',
 'last_optimized_date',
 'optimization_eligible_date',
 'is_published',
 'is_deleted']

## 1. My rule and its reason codes


Rule:
I rank content for refresh based on two signals: content staleness (days since last update) and search volume. Older content with higher search volume receives a higher refresh score because it has greater potential to benefit from updating.

Reason Codes:
• STALE_CONTENT – The content has not been updated for a long time.
• HIGH_SEARCH_VOLUME – The keyword has strong search demand.
• LOW_PRIORITY – The content does not currently meet the refresh threshold.

In [12]:
content_df["content_updated_date"] = pd.to_datetime(
    content_df["content_updated_date"],
    errors="coerce"
)

content_df["days_since_update"] = (
    pd.Timestamp.today().normalize() -
    content_df["content_updated_date"]
).dt.days

content_df[["content_updated_date", "days_since_update"]].head()

,content_updated_date,days_since_update
0,2026-07-01,28
1,2026-07-01,28
2,2026-07-01,28
3,2026-06-15,44
4,2026-06-01,58


In [15]:
# Signal Check 1

content_df["stale_bucket"] = pd.cut(
    content_df["days_since_update"],
    bins=[-1, 30, 90, 180, 10000],
    labels=[
        "Fresh (0-30)",
        "31-90",
        "91-180",
        "180+"
    ]
)

stale_check = (
    content_df
    .groupby("stale_bucket", observed=True)
    .size()
    .reset_index(name="n")
)

stale_check

,stale_bucket,n
0,Fresh (0-30),75214
1,31-90,299140
2,91-180,43330
3,180+,101922


In [16]:
# Signal Check 2 (Search Volume)

content_df["volume_bucket"] = pd.qcut(
    content_df["search_volume"],
    4,
    duplicates="drop"
)

volume_check = (
    content_df
    .groupby("volume_bucket", observed=True)
    .size()
    .reset_index(name="n")
)

print("\nSignal 2 - Search Volume")
print(volume_check)


Signal 2 - Search Volume
      volume_bucket       n
0    (-0.001, 10.0]  262477
1      (10.0, 20.0]   28824
2  (20.0, 368000.0]   85683


## 2. Build the ranked queue

The baseline score combines content staleness and search volume. Higher scores indicate a higher refresh priority. The ranked queue is exported as baseline_action_score.csv.

In [17]:
from pathlib import Path

baseline = content_df.copy()

# Normalise values
baseline["stale_score"] = (
    baseline["days_since_update"]
    / baseline["days_since_update"].max()
)

baseline["volume_score"] = (
    baseline["search_volume"]
    / baseline["search_volume"].max()
)

# Final baseline score
baseline["refresh_score"] = (
    0.6 * baseline["stale_score"]
    + 0.4 * baseline["volume_score"]
)

# Reason code
baseline["reason_code"] = np.where(
    baseline["days_since_update"] > 180,
    "STALE_CONTENT",
    "LOW_PRIORITY"
)

# Action
baseline["action_label"] = np.where(
    baseline["refresh_score"] >= 0.60,
    "REFRESH_CONTENT",
    "MONITOR"
)

# Rank
queue = baseline.sort_values(
    "refresh_score",
    ascending=False
)

# Save CSV
output_path = Path("work/outputs")
output_path.mkdir(parents=True, exist_ok=True)

queue.to_csv(
    output_path / "baseline_action_score.csv",
    index=False
)

print("CSV written successfully.")
queue.head()

CSV written successfully.


,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,is_published,is_deleted,days_since_update,stale_bucket,volume_bucket,stale_score,volume_score,refresh_score,reason_code,action_label
38499,client_73cda7b4e4f265ea,content_b9ffa30eb293951f,keyword_f7499e3a8e27232f,url_48cd7211902c4c57,28,3,98,2025-02-28,2026-05-20,keyword article,...,True,False,70,31-90,"(20.0, 368000.0]",0.109546,1.0,0.465728,LOW_PRIORITY,MONITOR
55776,client_a2eeb8899886adde,content_04e4047dc8eef2fd,keyword_df6aeefdfc9a1d6f,url_d3adb7edf174dc35,29,4,141,2026-01-27,2026-06-01,keyword article,...,True,False,58,31-90,"(20.0, 368000.0]",0.090767,1.0,0.454460,LOW_PRIORITY,MONITOR
308457,client_65de48885f4ef01b,content_7d523a38b93d7ed4,keyword_5b9ed3149d8d1703,url_f208390f2bbf8ad0,29,5,98,2025-05-31,2025-06-01,keyword article,...,True,False,423,180+,"(-0.001, 10.0]",0.661972,0.0,0.397183,STALE_CONTENT,MONITOR
308425,client_65de48885f4ef01b,content_7cd1b9802874c0e0,keyword_59163bfe1140d28e,url_7fcd9001e441843a,35,6,104,2025-05-31,2025-06-01,keyword article,...,True,False,423,180+,"(-0.001, 10.0]",0.661972,0.0,0.397183,STALE_CONTENT,MONITOR
241702,client_65de48885f4ef01b,content_b350dbb7dcf3c26e,keyword_2e3d997c7912061d,url_132cc8fe947d633a,33,5,102,2025-06-01,2025-06-01,keyword article,...,True,False,423,180+,"(-0.001, 10.0]",0.661972,0.0,0.397183,STALE_CONTENT,MONITOR


## 3. Top-20 review

The highest-ranked pages were reviewed manually. Each recommendation includes the proposed action, reason code, confidence level, and what could make the recommendation incorrect.

In [18]:
top20 = queue.head(20)

top20[
    [
        "content_hash_id",
        "refresh_score",
        "reason_code",
        "action_label",
        "days_since_update",
        "search_volume"
    ]
]

,content_hash_id,refresh_score,reason_code,action_label,days_since_update,search_volume
38499,content_b9ffa30eb293951f,0.465728,LOW_PRIORITY,MONITOR,70,368000.0
55776,content_04e4047dc8eef2fd,0.454460,LOW_PRIORITY,MONITOR,58,368000.0
308457,content_7d523a38b93d7ed4,0.397183,STALE_CONTENT,MONITOR,423,0.0
308425,content_7cd1b9802874c0e0,0.397183,STALE_CONTENT,MONITOR,423,0.0
241702,content_b350dbb7dcf3c26e,0.397183,STALE_CONTENT,MONITOR,423,0.0
241929,content_b768d6344b30b9d4,0.397183,STALE_CONTENT,MONITOR,423,0.0
241930,content_b769faef0ad77a2d,0.397183,STALE_CONTENT,MONITOR,423,0.0
241851,content_b5fef07169613b59,0.397183,STALE_CONTENT,MONITOR,423,0.0
241855,content_b615a83bda980ae8,0.397183,STALE_CONTENT,MONITOR,423,0.0
241869,content_b65a425817baab16,0.397183,STALE_CONTENT,MONITOR,423,0.0


## 4. Weak picks + leakage check

Weak Picks:
Some pages may rank highly simply because they are old, even if they are no longer valuable. Other pages may have high search volume but already satisfy user intent.

Leakage Check:
No future information or product flags were used. The rule only relies on historical signals available at scoring time.

In [19]:
print("Leakage Check")

print("Future window used: No")
print("Product flags used: No")
print("Signals used:")
print("- days_since_update")
print("- search_volume")

print("\nWeak Pick Examples")

weak = queue.tail(5)[
    [
        "content_hash_id",
        "refresh_score",
        "action_label"
    ]
]

print(weak)

Leakage Check
Future window used: No
Product flags used: No
Signals used:
- days_since_update
- search_volume

Weak Pick Examples
                 content_hash_id  refresh_score action_label
514438  content_2aa1fe5c3754b96e            NaN      MONITOR
514821  content_329987d42a85c7ed            NaN      MONITOR
515394  content_3e5c6320d7bece6a            NaN      MONITOR
516673  content_58c3bd2a9cb2feb3            NaN      MONITOR
519251  content_8e631e25ac195607            NaN      MONITOR


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.